In [4]:
!pip install pdfminer.six

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.6 MB 1.1 MB/s eta 0:00:05
   ----- ---------------------------------- 0.8/5.6 MB 1.1 MB/s eta 0:00:05
   ------- -------------------------------- 1.0/5.6 MB 1.1 MB/s eta 0:00:05
   --------- ------------------------------ 1.3/5.6 MB 1.1 MB/s eta 0:00:04
   ----------- ---------------------------- 1.6/5.6 MB 1.1 MB/s eta 0:00:04
   ------------- -------------------------- 1.8/5.6 MB 1.1 MB/s eta 0:00:04
   ------------- -------------------------- 1.8/5.6 MB 1.1 MB/s eta 0:00:04
   -------------- ------------------------- 2.1/5.6 MB 1.1 MB/s eta 0:00:04
   ---------------- ----------------------- 2.4/5.6 MB 1.1 MB/s eta 0:00:03
   ------------------ --------------------- 2.6/5.6 MB 1.1 MB/s eta 0:00:03
   -------------------- ---------

In [16]:
import os
import io
import re
import json
import torch
import spacy
from tqdm import tqdm
from keras.preprocessing.sequence import pad_sequences
from pytorch_pretrained_bert import BertTokenizer, BertForTokenClassification
from pdfminer.converter import TextConverter
from pdfminer.pdfinterp import PDFPageInterpreter, PDFResourceManager
from pdfminer.layout import LAParams
from pdfminer.pdfpage import PDFPage
from pdfminer.pdfparser import PDFSyntaxError
# Suppress TensorFlow warnings
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# Load spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully.")
# Tag mappings
tag2idx = {
    'I-NAME': 0, 'L-CLG': 1, 'L-SKILLS': 2, 'B-EMAIL': 3, 'L-YOE': 4, 'I-COMPANY': 5,
    'I-EMAIL': 6, 'I-DEG': 7, 'U-GRADYEAR': 8, 'L-EMAIL': 9, 'B-GRADYEAR': 10,
    'X': 11, 'L-NAME': 12, 'L-GRADYEAR': 13, 'I-LOC': 14, 'U-EMAIL': 15,
    '[SEP]': 16, 'B-CLG': 17, 'B-DESIG': 18, 'L-DEG': 19, 'B-YOE': 20,
    'U-YOE': 21, 'I-GRADYEAR': 22, 'U-CLG': 23, 'B-SKILLS': 24, 'B-LOC': 25,
    'B-COMPANY': 26, 'U-SKILLS': 27, 'L-DESIG': 28, 'U-COMPANY': 29, 'U-LOC': 30,
    'B-DEG': 31, 'U-NAME': 32, '[CLS]': 33, 'U-DEG': 34, 'U-DESIG': 35,
    'I-CLG': 36, 'O': 37, 'L-COMPANY': 38, 'B-NAME': 39, 'I-YOE': 40,
    'L-LOC': 41, 'I-SKILLS': 42, 'I-DESIG': 43
}
print("\nTag to Index Mapping:")
for tag, idx in tag2idx.items():
    print(f"{tag}: {idx}")
# Create reverse mapping
idx2tag = {v: k for k, v in tag2idx.items()}
def load_bert_model(model_path):
    """Load the BERT model for token classification"""
    print(f"\nLoading BERT model from {model_path}...")
    # Initialize the model with the same architecture
    model = BertForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(tag2idx))
    # Load the trained weights
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()  # Set to evaluation mode
    # Initialize tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)
    print("BERT model loaded successfully.")
    return model, tokenizer
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file"""
    print(f"\nExtracting text from PDF: {pdf_path}")
    text = ""
    
    try:
        with open(pdf_path, 'rb') as file:
            resource_manager = PDFResourceManager()
            for page in PDFPage.get_pages(file, caching=True, check_extractable=True):
                fake_file_handle = io.StringIO()
                converter = TextConverter(
                    resource_manager,
                    fake_file_handle,
                    codec='utf-8',
                    laparams=LAParams()
                )
                page_interpreter = PDFPageInterpreter(resource_manager, converter)
                page_interpreter.process_page(page)
                page_text = fake_file_handle.getvalue()
                text += page_text + "\n"
                # Close open handles
                converter.close()
                fake_file_handle.close()
        print(f"Extracted {len(text)} characters of text.")
        # Print first 500 characters as preview
        print("\nText preview:")
        print(text[:500] + "..." if len(text) > 500 else text)
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
    return text

def clean_text(text):
    """Clean raw text extracted from PDF."""
    text = text.replace('\u2022', ' ')  # Replace bullet points
    text = text.replace('\u2013', '-')  # Replace en-dash
    text = re.sub(r'\s+', ' ', text)    # Collapse multiple spaces
    return text.strip()


def extract_sentences(text):
    """Extract sentences from text using spaCy"""
    print("\nExtracting sentences from text...")
    doc = nlp(text)
    sentences = [sent for sent in doc.sents]
    print(f"Extracted {len(sentences)} sentences.")
    
    # Print first 3 sentences as preview
    print("\nSentence preview:")
    for i, sent in enumerate(sentences[:3]):
        print(f"Sentence {i+1}: {sent}")
    
    return sentences

def tokenize_sentences(sentences, tokenizer):
    """Properly tokenize sentences for BERT."""
    print("\nTokenizing sentences for BERT...")
    tokenized_texts = []
    original_tokens = []
    for sentence in sentences:
        words = [token.text for token in sentence if token.text.strip() != ""]
        # Skip empty sentences
        if len(words) == 0:
            continue
        tokens = ['[CLS]']
        orig_tokens = []
        for word in words:
            word_tokens = tokenizer.tokenize(word)
            if len(word_tokens) == 0:
                continue
            tokens.extend(word_tokens)
            orig_tokens.append(word)  # Save the original word
        tokens.append('[SEP]')
        tokenized_texts.append(tokens)
        original_tokens.append(orig_tokens)
    print(f"Tokenized {len(tokenized_texts)} sentences.")
    # Print preview
    if tokenized_texts:
        print("\nFirst tokenization preview:")
        print(f"Original: {original_tokens[0]}")
        print(f"Tokenized: {tokenized_texts[0]}")
    return tokenized_texts, original_tokens
def prepare_bert_inputs(tokenized_texts, tokenizer):
    """Prepare input IDs and attention masks for BERT"""
    print("\nPreparing BERT inputs...")
    MAX_LEN = 512
    # Convert tokens to IDs and pad sequences
    input_ids = pad_sequences(
        [tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
        maxlen=MAX_LEN, dtype="long", truncating="post", padding="post"
    )
    
    # Create attention masks
    attention_masks = [[float(i > 0) for i in ids] for ids in input_ids]
    
    print(f"Prepared input IDs and attention masks for {len(input_ids)} sentences.")
    print(f"Input shape: {input_ids.shape}")
    
    return input_ids, attention_masks

def run_bert_model(bert_model, input_ids, attention_masks):
    """Run BERT model for entity extraction"""
    print("\nRunning BERT model for entity extraction...")
    predictions_list = []
    
    with torch.no_grad():
        for i in range(len(input_ids)):
            input_tensor = torch.tensor([input_ids[i]])
            mask_tensor = torch.tensor([attention_masks[i]])
            
            # Get model outputs
            outputs = bert_model(input_tensor, attention_mask=mask_tensor)
            
            # Handle different output formats
            if isinstance(outputs, tuple):
                logits = outputs[0]
                print(f"Logits shape from tuple: {logits.shape}")
            else:
                logits = outputs
                print(f"Logits shape direct: {logits.shape}")
            
            # Apply argmax on the last dimension (class probabilities)
            if len(logits.shape) == 3:  # [batch_size, seq_len, num_labels]
                predictions = torch.argmax(logits, dim=2).detach().cpu().numpy()[0]
            elif len(logits.shape) == 2:  # [seq_len, num_labels]
                predictions = torch.argmax(logits, dim=1).detach().cpu().numpy()
            else:
                print(f"Unexpected logits shape: {logits.shape}")
                continue
                
            predictions_list.append(predictions)
    
    print(f"Generated predictions for {len(predictions_list)} sentences.")
    
    return predictions_list

def extract_token_entity_pairs(tokenized_texts, original_tokens, predictions, idx2tag):
    """Extract token-entity pairs from model predictions"""
    print("\nExtracting token-entity pairs from predictions...")
    token_entity_pairs = []
    
    for i, tokens in enumerate(tokenized_texts):
        j = 0
        token_idx = 0
        
        while j < len(tokens) and token_idx < len(original_tokens[i]):
            if j >= len(predictions[i]):
                break
                
            tag = idx2tag.get(predictions[i][j], 'O')
            
            # Skip special tokens
            if tag in ['[CLS]', '[SEP]']:
                j += 1
                continue
            
            # Skip subword tokens (X)
            if tag == 'X':
                j += 1
                continue
            
            # Skip non-entity tokens
            if tag == 'O':
                j += 1
                token_idx += 1
                continue
            
            # Extract entity type and position (B/I/L/U)
            if '-' in tag:
                position, entity_type = tag.split('-')
                
                # For single token entity (U)
                if position == 'U':
                    if token_idx < len(original_tokens[i]):
                        token_entity_pairs.append((original_tokens[i][token_idx], entity_type))
                    j += 1
                    token_idx += 1
                    continue
                
                # For multi-token entity (B-I-L)
                if position == 'B':
                    entity_text = original_tokens[i][token_idx] if token_idx < len(original_tokens[i]) else ""
                    entity_start_idx = token_idx
                    j += 1
                    token_idx += 1
                    
                    # Collect all I and L tokens
                    while j < len(tokens) and j < len(predictions[i]) and token_idx < len(original_tokens[i]):
                        next_tag = idx2tag.get(predictions[i][j], 'O')
                        
                        if next_tag.startswith('I-') and next_tag.endswith(entity_type):
                            entity_text += " " + original_tokens[i][token_idx]
                            j += 1
                            token_idx += 1
                        elif next_tag.startswith('L-') and next_tag.endswith(entity_type):
                            entity_text += " " + original_tokens[i][token_idx]
                            # Add the full entity
                            token_entity_pairs.append((entity_text, entity_type))
                            j += 1
                            token_idx += 1
                            break
                        else:
                            # If sequence is broken, still add what we have
                            token_entity_pairs.append((entity_text, entity_type))
                            break
                else:
                    j += 1
                    token_idx += 1
            else:
                j += 1
                token_idx += 1
    
    print(f"Extracted {len(token_entity_pairs)} token-entity pairs.")
    
    return token_entity_pairs

def group_entities_by_type(token_entity_pairs):
    """Group extracted entities by their type"""
    print("\nGrouping entities by type...")
    entity_groups = {}
    
    for token, entity_type in token_entity_pairs:
        if entity_type not in entity_groups:
            entity_groups[entity_type] = []
        
        # Only add unique entities
        if token not in entity_groups[entity_type]:
            entity_groups[entity_type].append(token)
    
    print(f"Found {len(entity_groups)} different entity types.")
    
    return entity_groups

def process_resume(pdf_path, bert_model, tokenizer):
    """Process a resume PDF and extract entities step by step"""
    # Step 1: Extract text from PDF
    text = extract_text_from_pdf(pdf_path)
    text = clean_text(text)
    if not text:
        print("Failed to extract text from PDF.")
        return None
    # Step 2: Extract sentences
    sentences = extract_sentences(text)
    # Step 3: Tokenize sentences
    tokenized_texts, original_tokens = tokenize_sentences(sentences, tokenizer)
    # Step 4: Prepare BERT inputs
    input_ids, attention_masks = prepare_bert_inputs(tokenized_texts, tokenizer)
    # Step 5: Run BERT model
    predictions_list = run_bert_model(bert_model, input_ids, attention_masks)
    # Step 6: Extract token-entity pairs
    token_entity_pairs = extract_token_entity_pairs(tokenized_texts, original_tokens, predictions_list, idx2tag)
    # Step 7: Group entities by type
    entity_groups = group_entities_by_type(token_entity_pairs)
    return text, token_entity_pairs, entity_groups

def main():
    """Main function to process a resume step by step"""
    # File paths
    bert_model_path = "eight_epoch.pt"
    resume_path = "ChResume.pdf"
    # Step 1: Load BERT model
    bert_model, tokenizer = load_bert_model(bert_model_path)
    # Step 2-7: Process resume
    text, token_entity_pairs, entity_groups = process_resume(resume_path, bert_model, tokenizer)
    if token_entity_pairs:
        # Display all extracted entities
        print("\n===== ALL EXTRACTED ENTITIES =====")
        for token, entity in token_entity_pairs:
            print(f"{token} => {entity}")
        # Display entities grouped by type
        print("\n===== ENTITIES BY TYPE =====")
        for entity_type, tokens in entity_groups.items():
            print(f"\n{entity_type}:")
            for i, token in enumerate(tokens):
                print(f"  {i+1}. {token}")
        # Save results to text file
        with open("resume_entities.txt", "w", encoding="utf-8") as f:
            f.write("===== ALL EXTRACTED ENTITIES =====\n")
            for token, entity in token_entity_pairs:
                f.write(f"{token} => {entity}\n")
            f.write("\n===== ENTITIES BY TYPE =====\n")
            for entity_type, tokens in entity_groups.items():
                f.write(f"\n{entity_type}:\n")
                for i, token in enumerate(tokens):
                    f.write(f"  {i+1}. {token}\n")
        print("\nEntities saved to resume_entities.txt")
    else:
        print("Failed to extract entities from resume.")

if __name__ == "__main__":
    main()

Loading spaCy model...
spaCy model loaded successfully.

Tag to Index Mapping:
I-NAME: 0
L-CLG: 1
L-SKILLS: 2
B-EMAIL: 3
L-YOE: 4
I-COMPANY: 5
I-EMAIL: 6
I-DEG: 7
U-GRADYEAR: 8
L-EMAIL: 9
B-GRADYEAR: 10
X: 11
L-NAME: 12
L-GRADYEAR: 13
I-LOC: 14
U-EMAIL: 15
[SEP]: 16
B-CLG: 17
B-DESIG: 18
L-DEG: 19
B-YOE: 20
U-YOE: 21
I-GRADYEAR: 22
U-CLG: 23
B-SKILLS: 24
B-LOC: 25
B-COMPANY: 26
U-SKILLS: 27
L-DESIG: 28
U-COMPANY: 29
U-LOC: 30
B-DEG: 31
U-NAME: 32
[CLS]: 33
U-DEG: 34
U-DESIG: 35
I-CLG: 36
O: 37
L-COMPANY: 38
B-NAME: 39
I-YOE: 40
L-LOC: 41
I-SKILLS: 42
I-DESIG: 43

Loading BERT model from eight_epoch.pt...
BERT model loaded successfully.

Extracting text from PDF: ChResume.pdf
Extracted 3082 characters of text.

Text preview:
CHARLOTTE MAY
Senior Data Scientist | CAP | DASCA

E

+1-713-645-3682



Email

q

www.linkedin.com/in/--lhamilton--



Phoenix, AZ

SUMMARY

TECH STACK

With eight years experience working as a data scientist and more than ten 
years of strong academic background